# 🏠 House Price Prediction — Bengaluru
**Goal:** predict the price of a house from its location, area, bedrooms and other features using regression.

**Tools:** Python · Pandas · Scikit-learn · Matplotlib

Steps in this notebook
1. Load the raw data
2. Clean the data and engineer features
3. Explore the data (EDA)
4. Train and compare 6 regression models
5. Evaluate the best model and make predictions

> The scripts in `src/` do exactly the same thing; this notebook is for explaining / demonstrating the project.

In [ ]:
import sys
sys.path.insert(0, "src")          # so we can import our own modules

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_prep import load_raw, clean_data, FEATURES, TARGET
from train import make_pipeline, MODELS, evaluate   # same pipeline used by train.py

pd.set_option("display.max_columns", 20)
%matplotlib inline

## 1. Load the raw data

In [ ]:
raw = load_raw()
print("Shape:", raw.shape)
raw.head(10)

In [ ]:
print("Missing values per column:")
print(raw.isna().sum())
print("\nDuplicate rows:", raw.duplicated().sum())
print("\nExamples of messy 'total_sqft' values:")
print(raw.loc[~raw["total_sqft"].str.replace(".", "", regex=False).str.isdigit(), "total_sqft"].head(8).tolist())

## 2. Data cleaning & feature engineering
* remove duplicates, drop `society` (too many missing values)
* `size` → `bhk` ("2 BHK" → 2), `total_sqft` → number (ranges averaged, units converted)
* `availability` → `ready_to_move` (1 / 0)
* rare locations (< 10 houses) grouped as **other**
* outliers removed: < 300 sq.ft per bedroom, bath > bhk + 2, unusual price-per-sq.ft for the location

In [ ]:
df = clean_data(raw)
df.head()

In [ ]:
df.describe().round(2)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["price"], bins=60)
axes[0].set(title="Price (₹ lakh) - right skewed", xlabel="Price (₹ lakh)")
axes[1].hist(np.log1p(df["price"]), bins=60)
axes[1].set(title="log(price) - close to normal", xlabel="log(1 + price)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df["total_sqft"], df["price"], s=8, alpha=0.3)
ax.set(title="Price vs total area", xlabel="Total area (sq.ft)", ylabel="Price (₹ lakh)")
plt.show()

In [ ]:
top = (df[df["location"] != "other"].groupby("location")["price_per_sqft"]
       .median().sort_values(ascending=False).head(10))
print("Most expensive locations (median ₹ per sq.ft):")
top.round(0)

In [ ]:
df[["total_sqft", "bhk", "bath", "balcony", "ready_to_move", "price"]].corr().round(2)

## 4. Train and compare regression models
80% of the data is used for training and 20% is kept aside for testing.
Every model uses the same pipeline: impute missing values → log(area) + scaling → one-hot encode location & area type → regressor trained on log(price).

Models are **ranked by 5-fold cross-validation on the training data**, which averages five different splits and is more reliable than a single test split. The test houses are only used for the final scores.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, train_test_split

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train):,} rows   Test: {len(X_test):,} rows")

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results = []
fitted = {}
for name, regressor in MODELS.items():
    pipe = make_pipeline(regressor)
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring="r2").mean()
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    scores = evaluate(y_test, pipe.predict(X_test))
    results.append({"Model": name, "CV_R2": cv_r2, **scores})

results = pd.DataFrame(results).sort_values("CV_R2", ascending=False).reset_index(drop=True)
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
r = results.sort_values("CV_R2")
ax.barh(r["Model"], r["CV_R2"])
ax.set(title="Cross-validated R² on the training data (higher is better)", xlabel="R²", xlim=(0.4, 1))
plt.show()

## 5. Best model — evaluation and predictions

In [ ]:
best_name = results.loc[0, "Model"]
best = fitted[best_name]
pred = best.predict(X_test)
print("Best model:", best_name)
for k, v in evaluate(y_test, pred).items():
    print(f"  {k:<13} {v:.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred, s=8, alpha=0.4)
lim = [0, max(y_test.max(), pred.max())]
ax.plot(lim, lim, color="tab:orange")
ax.set(title=f"Actual vs predicted - {best_name}", xlabel="Actual (₹ lakh)", ylabel="Predicted (₹ lakh)")
plt.show()

In [ ]:
new_houses = pd.DataFrame([
    {"location": "Whitefield",   "area_type": "Super built-up Area", "total_sqft": 1200, "bhk": 2, "bath": 2, "balcony": 1, "ready_to_move": 1},
    {"location": "Koramangala",  "area_type": "Super built-up Area", "total_sqft": 1650, "bhk": 3, "bath": 3, "balcony": 2, "ready_to_move": 1},
    {"location": "Electronic City", "area_type": "Built-up Area",    "total_sqft": 1000, "bhk": 2, "bath": 2, "balcony": 1, "ready_to_move": 0},
    {"location": "Hebbal",       "area_type": "Plot Area",           "total_sqft": 2400, "bhk": 4, "bath": 4, "balcony": 2, "ready_to_move": 1},
])
new_houses["predicted_price_lakh"] = best.predict(new_houses[FEATURES]).round(2)
new_houses

## Conclusion
* **Area** and **location** are by far the most important factors for house price.
* Predicting **log(price)** and using **log(area)** makes the price-area relationship nearly linear, so even simple linear models are competitive. Gradient Boosting ranks best in cross-validation, and a monotonic rule on the area keeps the price from falling when a house gets bigger.
* These are asking prices from real listings with few features, so an average error of about 19% is realistic. Treat an estimate as a guide, not a valuation.
* The final model (saved by `python src/train.py`) is used by the web app (`python app.py`) and the command-line tool (`python src/predict.py`).